# Multi-Factor Model Diagnostic

market + size + momentum + vol (generic over `risk_model.factors`)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dbutil import load_data
from config import get, BARS_PER_DAY

ANN = np.sqrt(BARS_PER_DAY * 365)          # 10min-bar -> annual

# Factor set is read from config so this notebook tracks the risk model.
FACTORS = get('risk_model.factors', ['market', 'size'])
FCOLS = [f'{f}_factor' for f in FACTORS]
BCOLS = [f'beta_{f}' for f in FACTORS]
LABELS = {'market': 'Market (EW)', 'size': 'Size (SMB)',
          'momentum': 'Momentum (WML)', 'vol': 'Vol (LMH)'}
flabel = lambda f: LABELS.get(f, f.capitalize())

factors = load_data('risk_factors')
factors['timestamp'] = pd.to_datetime(factors['timestamp'])
factors = factors.sort_values('timestamp').reset_index(drop=True)

loadings = load_data('factor_loadings')
loadings['date'] = pd.to_datetime(loadings['date'])

res = load_data('residual_returns',
                columns=['timestamp', 'symbol', 'raw_return', 'residual_return'])
res['timestamp'] = pd.to_datetime(res['timestamp'])

FPRESENT = [c for c in FCOLS if c in factors.columns]
BPRESENT = [c for c in BCOLS if c in loadings.columns]
print(f"configured factors : {', '.join(FACTORS)}")
print(f"present in risk_factors: {', '.join(FPRESENT)}  "
      f"(missing -> rebuild risk_model/factor_returns.py)")
print(f"risk_factors     : {len(factors):>10,} bars  "
      f"({factors['timestamp'].min().date()} -> {factors['timestamp'].max().date()})")
print(f"factor_loadings  : {len(loadings):>10,} rows, {loadings['symbol'].nunique()} symbols")
print(f"residual_returns : {len(res):>10,} rows, {res['symbol'].nunique()} symbols")

## 1. Factor Returns
Cumulative compounded return of each tradable factor, plus universe breadth.

In [ ]:
ts = factors.set_index('timestamp')
cum = (1 + ts[FPRESENT].fillna(0)).cumprod() - 1

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.68, 0.32],
                    vertical_spacing=0.06,
                    subplot_titles=('Cumulative factor return (%)', 'Universe members per bar'))
for col in FPRESENT:
    fig.add_trace(go.Scatter(x=cum.index, y=cum[col] * 100,
                             name=flabel(col.replace('_factor', ''))), row=1, col=1)
fig.add_trace(go.Scatter(x=ts.index, y=ts['n_members'], name='n_members',
                         line=dict(color='gray')), row=2, col=1)
fig.update_layout(height=620, title='Factor returns', template='plotly_white')
fig.show()

stats = []
for col in FPRESENT:
    f = col.replace('_factor', '')
    r = ts[col].dropna()
    mu, sd = r.mean(), r.std()
    stats.append({'factor': f, 'label': flabel(f),
                  'ann_return_%': mu * BARS_PER_DAY * 365 * 100,
                  'ann_vol_%': sd * ANN * 100,
                  'ann_sharpe': (mu / sd * ANN) if sd > 0 else np.nan,
                  'mean_bps_per_bar': mu * 1e4})
stats_df = pd.DataFrame(stats).set_index('factor')
stats_df.round(3)

## 2. Variance Reduction (acceptance gate)
The core check, mirroring `risk_model/residual_returns.run_acceptance_checks`:
residualization must remove systematic variance and the residual must not be
nearly identical to the raw return.

In [ ]:
min_vr = get('risk_model.acceptance.min_variance_reduction', 0.05)
max_corr = get('risk_model.acceptance.max_residual_raw_corr', 0.95)

sub = res.dropna(subset=['raw_return', 'residual_return'])
var_ratio = sub['residual_return'].var() / sub['raw_return'].var()
corr = sub['residual_return'].corr(sub['raw_return'])

print(f"var(residual)/var(raw) = {var_ratio:.4f}   (must be <= {1 - min_vr:.2f})"
      f"   {'PASS' if var_ratio <= 1 - min_vr else 'FAIL'}")
print(f"corr(residual, raw)    = {corr:.4f}   (must be <= {max_corr:.2f})"
      f"   {'PASS' if corr <= max_corr else 'FAIL'}")
print(f"systematic variance removed = {(1 - var_ratio) * 100:.1f}%")

g = sub.groupby('symbol')
per = (g['residual_return'].var() / g['raw_return'].var()).replace([np.inf, -np.inf], np.nan).dropna()
fig = go.Figure(go.Histogram(x=per.values, nbinsx=40))
fig.add_vline(x=1 - min_vr, line_dash='dash', line_color='red', annotation_text='gate')
fig.update_layout(title=f'Per-symbol var(residual)/var(raw)  (median {per.median():.3f})',
                  xaxis_title='variance ratio', height=380, template='plotly_white')
fig.show()

## 3. Residual Orthogonality to the Factors
What residualizing buys: the residual return should be ~uncorrelated with each
factor, while the raw return is not.

In [ ]:
m = sub.merge(factors[['timestamp'] + FPRESENT], on='timestamp', how='left').dropna()
orth = pd.DataFrame({
    f'vs {c}': [m['raw_return'].corr(m[c]), m['residual_return'].corr(m[c])]
    for c in FPRESENT
}, index=['raw_return', 'residual_return']).round(4)
# residual-vs-factor corr per factor (should be ~0) - reused in the summary
res_corr = {c: orth.loc['residual_return', f'vs {c}'] for c in FPRESENT}
orth

## 4. Factor Collinearity (correlation + VIF)

High pairwise correlation or VIF >> 1 inflates the variance of the OLS betas and
makes the neutrality constraints near-collinear. Watch this as factors are added
(momentum/vol vs market).

In [ ]:
F = factors.set_index('timestamp')[FPRESENT].dropna()
corr_m = F.corr()

# VIF per factor = 1 / (1 - R^2) of regressing each factor on the others.
vif = {}
X = F.values
for j, c in enumerate(FPRESENT):
    y = X[:, j]
    others = np.delete(X, j, axis=1)
    if others.shape[1] == 0:
        vif[c] = 1.0
        continue
    design = np.column_stack([np.ones(len(others)), others])
    beta, *_ = np.linalg.lstsq(design, y, rcond=None)
    r2 = 1 - (y - design @ beta).var() / y.var()
    vif[c] = 1.0 / (1.0 - r2) if r2 < 1 else np.inf

fig = go.Figure(go.Heatmap(z=corr_m.values, x=FPRESENT, y=FPRESENT,
                           zmin=-1, zmax=1, colorscale='RdBu',
                           text=corr_m.round(2).values, texttemplate='%{text}'))
fig.update_layout(title='Factor-return correlation', height=420, template='plotly_white')
fig.show()

# diagnostics reused in the summary
import numpy as _np
_off = corr_m.where(~_np.eye(len(FPRESENT), dtype=bool)).abs()
max_corr_off = float(_off.max().max()) if len(FPRESENT) > 1 else 0.0
max_vif = float(max(vif.values()))
print('VIF per factor (>5-10 is a concern):')
pd.DataFrame({'VIF': pd.Series(vif)}).round(3)

## 5. Betas & R²

In [ ]:
latest = loadings.sort_values('date').groupby('symbol').tail(1)
cols_plot = BPRESENT + ['r_squared']
fig = make_subplots(rows=1, cols=len(cols_plot), subplot_titles=cols_plot)
for j, c in enumerate(cols_plot):
    fig.add_trace(go.Histogram(x=latest[c], nbinsx=30), row=1, col=j + 1)
fig.update_layout(height=340, showlegend=False, template='plotly_white',
                  title='Latest betas & R^2 across symbols')
fig.show()
latest[cols_plot].describe().round(3)

## 6. Beta Stability Over Time

In [ ]:
top = loadings['symbol'].value_counts().head(6).index.tolist()
fig = make_subplots(rows=len(BPRESENT), cols=1, shared_xaxes=True,
                    vertical_spacing=0.04, subplot_titles=BPRESENT)
for s in top:
    d = loadings[loadings['symbol'] == s].sort_values('date')
    for j, c in enumerate(BPRESENT):
        fig.add_trace(go.Scatter(x=d['date'], y=d[c], name=s, legendgroup=s,
                                 showlegend=(j == 0)), row=j + 1, col=1)
fig.update_layout(height=240 * max(len(BPRESENT), 1), template='plotly_white',
                  title='Beta stability over time (top-coverage symbols)')
fig.show()

## 7. Summary

In [ ]:
print('=' * 64)
print('MULTI-FACTOR MODEL - DIAGNOSTIC SUMMARY')
print(f"factors: {', '.join(FACTORS)}")
print('=' * 64)
print(f"systematic variance removed   : {(1 - var_ratio) * 100:5.1f}%   "
      f"({'PASS' if var_ratio <= 1 - min_vr else 'FAIL'})")
print(f"corr(residual, raw)           : {corr:6.3f}   "
      f"({'PASS' if corr <= max_corr else 'FAIL'})")
for c in FPRESENT:
    print(f"residual vs {c:18s} corr: {res_corr[c]:+.4f}   (~0 expected)")
print(f"max factor-pair |corr|        : {max_corr_off:.3f}")
print(f"max factor VIF                : {max_vif:6.2f}   (>5-10 = collinear)")
print(f"median per-symbol R^2         : {latest['r_squared'].median():.3f}")
for f in FACTORS:
    if f in stats_df.index:
        print(f"{stats_df.loc[f, 'label']:16s} ann. Sharpe : {stats_df.loc[f, 'ann_sharpe']:6.2f}")